# 📖 Notebook 3: The Saga Pattern

When a multi-step process fails halfway through, you need to **undo** the completed steps.
This is the Saga Pattern — one of the most important patterns in distributed systems.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why distributed transactions are hard (and why 2PC doesn't scale)
- How the Saga Pattern provides eventual consistency through compensations
- How to implement a Saga in Temporal with try/except and compensation lists
- How to test both the happy path and failure + rollback scenarios

## 🛠️ Setup

Make sure Temporal is running:

```bash
cd enterprise-patterns/long-running-jobs-temporal
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import asyncio
import uuid
from datetime import timedelta
from temporalio import activity, workflow
from temporalio.client import Client
from temporalio.worker import Worker
from temporalio.common import RetryPolicy

client = await Client.connect("localhost:7233")
print("✅ Connected to Temporal")

TASK_QUEUE = "saga-task-queue"

## 🤔 The Problem: Distributed Transactions

In a traditional database, you can wrap multiple operations in a **transaction**:

```sql
BEGIN TRANSACTION;
  INSERT INTO orders ...;
  UPDATE inventory SET quantity = quantity - 1;
  INSERT INTO payments ...;
COMMIT;  -- all succeed or all fail
```

If any statement fails, the database rolls back ALL changes. Simple!

But in a **distributed system**, each step talks to a different service:

```
Step 1: Order Service   → Create order        (own database)
Step 2: Payment Service → Charge credit card   (Stripe API)
Step 3: Inventory Svc   → Reserve items        (warehouse DB)
Step 4: Shipping Svc    → Schedule delivery    (FedEx API)
```

There's **no single transaction** that spans all four services. If Step 3 fails after
Steps 1 and 2 succeed, you can't just `ROLLBACK` — Stripe already charged the card!

### The Saga Solution

A **Saga** is a sequence of steps where each step has a **compensation** (undo action):

```
Step 1: Create order       → Compensation: Cancel order
Step 2: Charge payment     → Compensation: Refund payment
Step 3: Reserve inventory  → Compensation: Release inventory
Step 4: Ship order         → (no compensation needed if saga completes)
```

If Step 3 fails:
1. Run compensation for Step 2 → **Refund the payment**
2. Run compensation for Step 1 → **Cancel the order**
3. Report the failure

Compensations run in **reverse order** (last completed step first).

## 🏗️ Step 1: Define the Activities

Each step in our order process is an activity. For each step that changes state,
we also define a compensation activity that undoes it.

We'll use a shared `order_log` list to track what happened (in a real system,
this would be database records and API calls).

In [ ]:
# Shared log so we can see what happened in order
order_log = []

# Control whether inventory reservation should fail (for testing)
FORCE_INVENTORY_FAILURE = False


# ── Forward Activities (the actual steps) ──────────────────────

@activity.defn
async def create_order(order_id: str) -> dict:
    """Step 1: Create the order in our database."""
    activity.logger.info(f"Creating order {order_id}")
    await asyncio.sleep(0.3)
    order_log.append(f"✅ Order {order_id} created")
    return {"order_id": order_id, "status": "created"}


@activity.defn
async def charge_payment(order_id: str) -> dict:
    """Step 2: Charge the customer's credit card."""
    activity.logger.info(f"Charging payment for {order_id}")
    await asyncio.sleep(0.5)
    order_log.append(f"✅ Payment charged for {order_id}")
    return {"order_id": order_id, "payment_id": f"PAY-{order_id}", "amount": 99.99}


@activity.defn
async def reserve_inventory(order_id: str) -> dict:
    """Step 3: Reserve items in the warehouse."""
    activity.logger.info(f"Reserving inventory for {order_id}")
    await asyncio.sleep(0.3)

    if FORCE_INVENTORY_FAILURE:
        order_log.append(f"❌ Inventory reservation FAILED for {order_id}")
        raise RuntimeError(f"Warehouse API error: items out of stock for {order_id}")

    order_log.append(f"✅ Inventory reserved for {order_id}")
    return {"order_id": order_id, "warehouse": "WH-EAST", "reserved": True}


@activity.defn
async def schedule_shipping(order_id: str) -> dict:
    """Step 4: Schedule delivery with shipping provider."""
    activity.logger.info(f"Scheduling shipping for {order_id}")
    await asyncio.sleep(0.3)
    order_log.append(f"✅ Shipping scheduled for {order_id}")
    return {"order_id": order_id, "tracking": f"TRACK-{order_id}"}


# ── Compensation Activities (the undo steps) ──────────────────

@activity.defn
async def cancel_order(order_id: str) -> dict:
    """Undo Step 1: Cancel the order."""
    activity.logger.info(f"COMPENSATING: Cancelling order {order_id}")
    await asyncio.sleep(0.2)
    order_log.append(f"🔙 Order {order_id} CANCELLED (compensation)")
    return {"order_id": order_id, "status": "cancelled"}


@activity.defn
async def refund_payment(order_id: str) -> dict:
    """Undo Step 2: Refund the payment."""
    activity.logger.info(f"COMPENSATING: Refunding payment for {order_id}")
    await asyncio.sleep(0.3)
    order_log.append(f"🔙 Payment REFUNDED for {order_id} (compensation)")
    return {"order_id": order_id, "refund_id": f"REF-{order_id}"}


@activity.defn
async def release_inventory(order_id: str) -> dict:
    """Undo Step 3: Release the reserved inventory."""
    activity.logger.info(f"COMPENSATING: Releasing inventory for {order_id}")
    await asyncio.sleep(0.2)
    order_log.append(f"🔙 Inventory RELEASED for {order_id} (compensation)")
    return {"order_id": order_id, "released": True}


@activity.defn
async def send_notification(message: str) -> str:
    """Send a notification (email, push, etc.)."""
    activity.logger.info(f"Notification: {message}")
    await asyncio.sleep(0.1)
    order_log.append(f"📧 Notification: {message}")
    return f"sent: {message}"


print("✅ Defined 7 activities:")
print("   Forward:      create_order, charge_payment, reserve_inventory, schedule_shipping")
print("   Compensation: cancel_order, refund_payment, release_inventory")
print("   Utility:      send_notification")

## 🏗️ Step 2: The Saga Workflow

The Saga Workflow follows this pattern:

```
compensations = []          # empty list

try:
    do_step_1()
    compensations.append(undo_step_1)   # remember how to undo

    do_step_2()
    compensations.append(undo_step_2)

    do_step_3()              # if this fails...
    compensations.append(undo_step_3)

except Exception:
    # ...run compensations in REVERSE order
    for compensation in reversed(compensations):
        compensation()       # undo_step_2(), then undo_step_1()
```

The key insight: we build up the compensation list as we go. If step 3 fails,
only steps 1 and 2 have compensations — step 3 never added one because it didn't succeed.

In [ ]:
@workflow.defn
class OrderSagaWorkflow:
    """
    Order processing workflow using the Saga pattern.

    If any step fails, all previously completed steps are compensated
    (undone) in reverse order.
    """

    @workflow.run
    async def run(self, order_id: str) -> dict:
        # This list accumulates compensation actions as steps succeed.
        compensations = []

        retry = RetryPolicy(
            initial_interval=timedelta(seconds=1),
            backoff_coefficient=2.0,
            maximum_attempts=3,
        )

        try:
            # ── Step 1: Create Order ────────────────────────────
            order = await workflow.execute_activity(
                create_order, order_id,
                start_to_close_timeout=timedelta(seconds=10),
                retry_policy=retry,
            )
            compensations.append(cancel_order)  # remember the undo

            # ── Step 2: Charge Payment ─────────────────────────
            payment = await workflow.execute_activity(
                charge_payment, order_id,
                start_to_close_timeout=timedelta(seconds=30),
                retry_policy=retry,
            )
            compensations.append(refund_payment)

            # ── Step 3: Reserve Inventory ──────────────────────
            inventory = await workflow.execute_activity(
                reserve_inventory, order_id,
                start_to_close_timeout=timedelta(seconds=10),
                retry_policy=retry,
            )
            compensations.append(release_inventory)

            # ── Step 4: Schedule Shipping ──────────────────────
            shipping = await workflow.execute_activity(
                schedule_shipping, order_id,
                start_to_close_timeout=timedelta(seconds=10),
                retry_policy=retry,
            )

            # 🎉 All steps succeeded!
            await workflow.execute_activity(
                send_notification,
                f"Order {order_id} completed! Tracking: {shipping['tracking']}",
                start_to_close_timeout=timedelta(seconds=10),
            )

            return {
                "status": "completed",
                "order": order,
                "payment": payment,
                "inventory": inventory,
                "shipping": shipping,
            }

        except Exception as e:
            # ── SAGA COMPENSATION ──────────────────────────────
            # Something failed! Run compensations in REVERSE order.
            workflow.logger.error(f"Order {order_id} failed: {e}")
            workflow.logger.info(f"Running {len(compensations)} compensations...")

            for compensation in reversed(compensations):
                try:
                    await workflow.execute_activity(
                        compensation, order_id,
                        start_to_close_timeout=timedelta(seconds=10),
                        retry_policy=retry,
                    )
                except Exception as comp_err:
                    # Log but continue — we want to run ALL compensations
                    workflow.logger.error(
                        f"Compensation {compensation} failed: {comp_err}"
                    )

            await workflow.execute_activity(
                send_notification,
                f"Order {order_id} failed and was rolled back: {e}",
                start_to_close_timeout=timedelta(seconds=10),
            )

            return {"status": "rolled_back", "error": str(e)}


print("✅ Defined OrderSagaWorkflow with 4 steps and 3 compensations")

## ✅ Test 1: The Happy Path

Let's run the workflow when everything succeeds. All 4 steps should complete
with no compensations needed.

In [ ]:
# All activities for the worker
all_activities = [
    create_order, charge_payment, reserve_inventory, schedule_shipping,
    cancel_order, refund_payment, release_inventory, send_notification,
]

# Reset state
order_log.clear()
FORCE_INVENTORY_FAILURE = False


async def run_saga(order_id: str) -> dict:
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[OrderSagaWorkflow],
        activities=all_activities,
    ):
        return await client.execute_workflow(
            OrderSagaWorkflow.run,
            order_id,
            id=f"saga-{order_id}-{uuid.uuid4()}",
            task_queue=TASK_QUEUE,
        )


result = await run_saga("ORD-100")

print("📋 Execution Log:")
for entry in order_log:
    print(f"   {entry}")
print(f"\n📬 Final Result: {result['status']}")
print(f"   Tracking: {result['shipping']['tracking']}")

## ❌ Test 2: Failure + Rollback

Now let's force Step 3 (inventory reservation) to fail. Watch how the saga
automatically **undoes** Steps 1 and 2:

```
Step 1: Create order       ✅ (done)
Step 2: Charge payment     ✅ (done)
Step 3: Reserve inventory  ❌ (FAILS!)

→ Compensation: Refund payment     🔙
→ Compensation: Cancel order       🔙
→ Send failure notification        📧
```

In [ ]:
# Force inventory to fail
order_log.clear()
FORCE_INVENTORY_FAILURE = True

result = await run_saga("ORD-FAIL")

print("📋 Execution Log (notice the reverse compensation order):")
print("=" * 60)
for i, entry in enumerate(order_log, 1):
    print(f"   {i}. {entry}")

print(f"\n📬 Final Result: {result['status']}")
print(f"   Error: {result['error']}")
print()
print("💡 Key observations:")
print("   1. Steps 1 & 2 succeeded, Step 3 failed")
print("   2. Compensations ran in REVERSE: refund first, then cancel order")
print("   3. The customer was refunded — no money lost!")
print("   4. A notification was sent about the failure")

# Reset for future cells
FORCE_INVENTORY_FAILURE = False

## 🎨 Visualizing the Saga

Here's what the happy path and failure path look like side by side:

```
HAPPY PATH                        FAILURE PATH
──────────                        ────────────
create_order ──────✅             create_order ──────✅
    │                                 │
charge_payment ────✅             charge_payment ────✅
    │                                 │
reserve_inventory ─✅             reserve_inventory ─❌ FAIL!
    │                                 │
schedule_shipping ─✅             ┌───┴───────────────┐
    │                             │  COMPENSATIONS     │
send_notification ─📧            │  (reverse order)   │
    │                             │                    │
    ✅ COMPLETED                  │  refund_payment 🔙 │
                                  │  cancel_order  🔙 │
                                  └────────────────────┘
                                          │
                                  send_notification 📧
                                          │
                                  🔙 ROLLED BACK
```

Check the Temporal UI at http://localhost:8080 — click on the failed workflow
to see the full event history, including each compensation activity.

## 🔑 Idempotency: Making Compensations Safe

In production, compensations might be retried (network issues, timeouts). A refund
compensation must be **idempotent** — calling it twice should not refund twice.

The standard approach is **idempotency keys**:

```python
@activity.defn
async def refund_payment(order_id: str) -> dict:
    # Use order_id as the idempotency key.
    # Stripe's API supports this natively:
    #   stripe.Refund.create(
    #       payment_intent="pi_xxx",
    #       idempotency_key=f"refund-{order_id}"
    #   )
    # If this exact refund was already processed, Stripe returns
    # the existing refund instead of creating a new one.
    pass
```

**Rule of thumb**: Every activity that changes state should accept an idempotency key
so that retries are safe.

## ⚖️ Sagas vs Two-Phase Commit (2PC)

You might wonder: "Why not just use distributed transactions (2PC)?"

| | Saga | Two-Phase Commit (2PC) |
|---|------|------------------------|
| **Consistency** | Eventual (compensate on failure) | Strong (all or nothing) |
| **Availability** | High (each service independent) | Low (coordinator is bottleneck) |
| **Performance** | Fast (no locks held) | Slow (locks held during prepare phase) |
| **Complexity** | Must write compensations | Must implement 2PC protocol |
| **Scalability** | Scales well | Poor (locks + coordinator) |
| **Real-world use** | Netflix, Uber, Stripe | Traditional databases only |

**In microservices, Sagas win.** 2PC requires all services to support a locking protocol,
which external APIs (Stripe, FedEx) don't offer. Sagas work with any service.

## 📚 Summary

### What We Learned

1. **Distributed transactions** can't use a single database ROLLBACK
2. **The Saga pattern** uses compensation actions to undo completed steps
3. **Compensations run in reverse order** — last completed step is undone first
4. **Temporal makes Sagas easy** — just use try/except and a compensations list
5. **Idempotency keys** make compensations safe to retry
6. **Sagas provide eventual consistency** — the right trade-off for microservices

### The Pattern in 6 Lines

```python
compensations = []
try:
    await do_step()
    compensations.append(undo_step)
except Exception:
    for comp in reversed(compensations):
        await comp()
```

### Next Up

In **Notebook 4**, we'll explore advanced Temporal patterns: **child workflows** for
breaking up complex processes, **signals** for human-in-the-loop approvals, and
**timers** for scheduled delays.